In [ ]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"
# import threading
mt5.initialize()


def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)


def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M1':mt5.TIMEFRAME_M1, 'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['ema1'] =rates_frame['close'].ewm(span=9, adjust=False).mean()
    rates_frame['ema2'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
    rates_frame['ema3'] =rates_frame['close'].ewm(span=50, adjust=False).mean()

#     rates_frame['ema'] =ema(rates_frame['close'], 9)
#     rates_frame['ema'] =ema(rates_frame['close'], 9)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)
    
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
#         print(rates_frame.head())
    # Calculate Supertren
    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [ ]:
#NEED TO TEST MORE
#If three candles in H" USDJPY of one direction rest should follow,
#VERY GOod setup


# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "USDJPY"
a = get_values(symbol, 5000, 2, 'H2')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0


# 
for i in range(10, len(a)-10):
    if check==0:
        if direction(a, i) ==0 and direction(a, i-1) ==0 and direction(a, i-2) == 0 and a.iloc[i-2].open > a.iloc[i].ema:
            print("SELL")
            print("+++"*20)
            print(f"{a.iloc[i].name} -- ")
            buy_price = a.iloc[i].close
            check=1
            
        elif direction(a, i) ==1 and direction(a, i-1) ==1 and direction(a, i-2) == 1 and  a.iloc[i-2].open < a.iloc[i].ema:
            print("SELL")
            print("+++"*20)
            print(f"{a.iloc[i].name} -- ")
            buy_price = a.iloc[i].close
            check=2
            
    elif check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10)
        print(f"{pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >-5 and pp <0:
            continue
        elif pp>0:
            profit.append(pp)
            check=0
        elif pp <-15:
            profit.append(pp)
#             if pp<-50:
#                 profit.append(-50)
#             else:
#                 profit.append(pp)
            check=0

    elif check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >-5 and pp < 0:
            continue
        elif pp>0:
            profit.append(pp)
            check=0
        elif pp <-15:
            profit.append(pp)
#             if pp<-50:
#                 profit.append(-50)
#             else:
#                 profit.append(pp)
            check=0

In [ ]:
# Minute 1
#GOOD SETUP
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 40000, 5, 'M5')
timezone =pytz.timezone('Etc/GMT-2')
lot = 0.1

# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

def loss():
    return price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL)
for i in range(10, len(a)-5):
        
    if check==0:        
        if a.iloc[i].close < a.iloc[i].ema1 and a.iloc[i].close < a.iloc[i].ema2 and direction(a, i) == 0 and \
        a.iloc[i-1].close < a.iloc[i-1].ema1 and a.iloc[i-1].close < a.iloc[i-1].ema2 and direction(a, i-1) == 0 and \
        a.iloc[i-2].close < a.iloc[i-2].ema1 and a.iloc[i-2].close < a.iloc[i-2].ema2 and direction(a, i-2) == 0 and \
        price_action(symbol, lot, a.iloc[i].close, a.iloc[i].ema1, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10)) > -10.0 and \
        price_action(symbol, lot, a.iloc[i].close, a.iloc[i].ema2, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10)) > -10.0:
            print("=="*20)
            print(f"{a.iloc[i].name} !! {a.iloc[i].ema1} !! {a.iloc[i].ema2} !! {a.iloc[i].close}")
            c = 0
            buy_price = a.iloc[i].close
            pp_old = 0.0
            check=1
            checks = 0
            up = 0
            hpp = 0.0
            
#         if "00:05" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close > a.iloc[i].open:
#             print("=="*20)
#             print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2}")
#             c = 0
#             buy_price = a.iloc[i].close
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        lot = 0.1
        if c!=0:
            pp_old = pp
            up=1
#             print(f"up===>{up}")
        c+=1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
        ppopen = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        if hpp < pp_old:
            hpp = pp_old
        if pp > 3.0:
            up=1
            
        if pp>=10.0:
#             print(f"PP {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             profit.append(pp)
            checks=1
        if a.iloc[i].rsi1 < 10 and a.iloc[i].rsi1 > a.iloc[i-1].rsi1 and (a.iloc[i-1].rsi1 - a.iloc[i].rsi1) >=1:
            print(f"rsipp {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             if pp > 0.0:
#                 profit.append(pp)
#             elif pp<0.0 and up==1:
#                 profit.append(-1)
#             else:
#                 profit.append(pp)
            profit.append(pp)
            check=0
            continue
        if sell_price > a.iloc[i-1].ema1 + 20 or sell_price > a.iloc[i-1].ema2+ 20:
            print(f"ema_cross {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            if pp < -10:
                if up !=1:
                    profit.append(-10)
                else:
                    profit.append(-1)
            else:
                if pp > 0.0:
                    profit.append(pp)
                elif pp<0.0 and up==1:
                    profit.append(-1)
                else:
                    profit.append(pp)
            check=0
            continue
        if pp < hpp/1.75 and checks!=0:
            print(f"HPP_old {hpp/1.75}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(hpp/2)
            check =0
            continue
            


In [ ]:
#VERY POOR but test once
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 2000, 25, 'M5')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=1
            
        if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [ ]:
# Minute 1
#USDCAD M30 rsi7 and 14 peaks then falls sharply

#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "USDCAD"
a = get_values(symbol, 20000, 25, 'M30')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(10, len(a)-10):
    if check==0:
        if a.iloc[i].rsi1 <= 70 and a.iloc[i-1].rsi1 >= 70 and a.iloc[i].close < a.iloc[i].ema3:
            print("=="*20)
            print(f"{a.iloc[i].name}--{a.iloc[i].rsi2}")
            buy_price = a.iloc[i].close
            check=1
            
#         if a.iloc[i].rsi1 >= 70 and a.iloc[i-1].rsi1 >= a.iloc[i].rsi1:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.5, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 1.50
        print(f"{pp}--{ a.iloc[i].rsi2}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 40.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-35:
            profit.append(pp)
            check=0
#         else:
#             profit.append(pp)
        

    elif check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-5:
            profit.append(-5)
        else:
            profit.append(pp)
        check=0